In [2]:
import os
import torch
from internvl_utils import *
from transformers import AutoModel, AutoTokenizer
from PIL import Image

/home/emogenai4e/anaconda3/envs/hungvad/lib/python3.9/site-packages/torch/cuda/__init__.py:51: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/emogenai4e/anaconda3/envs/hungvad/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
mllm_path = "ppxin321/HolmesVAU-2B"

In [4]:
def load_model(mllm_path, device):
    model = AutoModel.from_pretrained(
        mllm_path,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        use_flash_attn=True,
        trust_remote_code=True,
        ).eval()
    tokenizer = AutoTokenizer.from_pretrained(mllm_path, trust_remote_code=True, use_fast=False)
    model = model.to(device)
    generation_config = dict(max_new_tokens=1024, do_sample=False)
    return model, tokenizer, generation_config

In [5]:
model, tokenizer, generation_config = load_model(mllm_path, device)

/home/emogenai4e/anaconda3/envs/hungvad/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


/home/emogenai4e/anaconda3/envs/hungvad/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [6]:
INPUT_SIZE = 448
MAX_NUM = 1

In [7]:
image_path = "/home/emogenai4e/emo/VHung/v1_71.jpg"
image = Image.open(image_path).convert("RGB")

transform = build_transform(input_size=INPUT_SIZE)
img = dynamic_preprocess(image, image_size=INPUT_SIZE, use_thumbnail=True, max_num=MAX_NUM)

In [8]:
def get_pixel_values_single_image(image_path, input_size=448, max_num=1):
    transform = build_transform(input_size=input_size)
    image = Image.open(image_path).convert('RGB')
    img = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    print("Number of tiles:", len(img))
    pixel_values = [transform(tile) for tile in img]
    pixel_values = torch.stack(pixel_values)
    num_patches_list = [pixel_values.shape[0]]
    return pixel_values, num_patches_list

pixel_values, num_patches_list = get_pixel_values_single_image(image_path, input_size=INPUT_SIZE, max_num=MAX_NUM)
print("pixel_values shape:", pixel_values.shape)
print("num_patches_list:", num_patches_list)

Number of tiles: 1
pixel_values shape: torch.Size([1, 3, 448, 448])
num_patches_list: [1]


In [9]:
def get_image_features(model, pixel_values, device):
    with torch.no_grad():
        pixel_values = pixel_values.to(torch.bfloat16)
        pixel_values = pixel_values.to(device)
        vit_embeds = model.vision_model(pixel_values=pixel_values, output_hidden_states=False, return_dict=True).last_hidden_state
        cls_token = vit_embeds[:, 0, :] 
    
    print("cls_token shape:", cls_token.shape)
    return cls_token
    

In [10]:
cls_token = get_image_features(model, pixel_values, device)

cls_token shape: torch.Size([1, 1024])


In [11]:
print(model.vision_model.config.hidden_size)
# hoặc
print(model.vision_model.config)

1024
InternVisionConfig {
  "architectures": [
    "InternVisionModel"
  ],
  "attention_dropout": 0.0,
  "drop_path_rate": 0.0,
  "dropout": 0.0,
  "hidden_act": "gelu",
  "hidden_size": 1024,
  "image_size": 448,
  "initializer_factor": 1.0,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_norm_eps": 1e-06,
  "model_type": "intern_vit_6b",
  "norm_type": "layer_norm",
  "num_attention_heads": 16,
  "num_channels": 3,
  "num_hidden_layers": 24,
  "patch_size": 14,
  "qk_normalization": false,
  "qkv_bias": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.37.2",
  "use_bfloat16": true,
  "use_flash_attn": true
}

